<a href="https://colab.research.google.com/github/YefridC09/ST-554-Project1-Template/blob/main/Task1/st554-project1-task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Title: ST-554 Project 1 Task 1 \
Author: Stephen Griggs \
Date: 2/11/2026

In [1]:
!pip install ucimlrepo

In [33]:
from ucimlrepo import fetch_ucirepo
import numpy as np

Remove any observations where the C6H6(GT) or CO(GT) are -200 as these represent missing values (which
we’ll ignore).

In [3]:
air_quality = fetch_ucirepo(id=360)
air_quality = air_quality.data.features
air_quality = air_quality[
    (air_quality["C6H6(GT)"] != -200) &
    (air_quality["CO(GT)"] != -200)
]

Response variable: C6H6(GT) \
Loss function
$$
  \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - c)^2}
$$

1. One where we don't consider any data other than the y's. That is, c is going to just be a constant that minimizes the function.
*Note: the calculus based answer for this comes out to be the sample mean, ȳ

2. One where we consider a linear equation (the SLR model) of one other numeric variable.
* Relating to the above, for each observation (i) our prediction is given by $c_i = b_0 + b_1 x_i$.
* Here xi is one of the other numeric variables from our data set. We’ll use the PT08.S1(CO)
variable.
* Note: the calculus based answer for this comes out to be the usual simple linear regression
estimates, which you can find using scipy.stats as done in the Fitting and Evaluating SLR
Models notes!

Using a Grid Search Algorithm

You'll be implementing a grid search to find the optimal value of $c$ based off of our data set.

Just $y$: Pseudo code for using just the $y$'s and no other variables:

1. Create a grid of values for $c$. Look at the first and third quartiles for the $\mathrm{C6H6(GT)}$ variable to consider reasonable values for $c$.

2. Create a squared error loss function that takes in $y$ and $c$ that outputs $(y - c)^2$.

3. Create a root mean squared error (objective) function that takes in $y$ and $c$ that outputs $\sqrt{ \frac{1}{n} \sum_{i=1}^{n} (y_i - c)^2 }$ (You can put step 2 and 3 into one function if you want.)

4. Use a list comprehensive to loop over the grid of $c$ values, finding the RMSE for each value of $c$.

5. Determine which value of $c$ gives the optimal (smallest) RMSE.

6. Report that as the prediction!

7. Wrap the above into a function that takes in a column of data and outputs the value of $c$.

- Run your algorithm on the $\mathrm{C6H6(GT)}$ variable and determine the \textbf{optimal} constant prediction.

- Just to make sure your algorithm generalizes, run it using the $\mathrm{PT08.S1(CO)}$ variable as the response (this shouldn't involve any new functions, just new function calls!). Be sure to adjust your grid accordingly (use the quantiles of this variable)!

In [4]:
# root mean square error function
def calc_rmse(response, c):
    return np.sqrt(1/len(response) * sum((response-c)**2))

def find_best_c(response, num_points=100):
    # q1 and q3 of response column
    q1, q3 = response.quantile([0.25, 0.75])
    grid = np.linspace(q1, q3, num_points)

    # loop over grid of c values finding rmse for each
    rmse_vals = [calc_rmse(response, c) for c in grid]
    paired = list(zip(grid, rmse_vals))

    # determine which value of c gives the smallest rmse
    best_c, min_rmse = min(paired, key=lambda x: x[1])

    # report best value of c as a prediction
    print(f"The best value of c is {best_c} with an RMSE of {min_rmse}")

    # return best value of c
    return best_c

In [5]:
find_best_c(air_quality["C6H6(GT)"])
find_best_c(air_quality["PT08.S1(CO)"])
np.mean(air_quality["C6H6(GT)"])
np.mean(air_quality["PT08.S1(CO)"])

The best value of c is 10.282828282828284 with an RMSE of 7.440564471926774
The best value of c is 1109.6363636363635 with an RMSE of 218.6684818310563


np.float64(1110.5807461873637)

Using y and another numeric variable x: \
Next, you'll implement the grid search to find the optimal pair of values for b0 and b1 using PT08.S1(CO) as your x variable and C6H6(GT) as your y variable. The pseudo code is very similar to that above, but your grid now has two-dimensions!
* You'll need to populate a grid of b0 and b1 values that you want to consider.
  * Use b0 values from -25 to -15 with increments of 0.1.
  * Use b1 values from -5 to 5 with increments of 0.01
* Report your optimal b0 and b1 combination.
* Create a function that takes in an x and and y column and outputs the optimal values.
* Then use these values to predict a new C6H6(GT) for a PT08.S1(CO) of 946, 1075, and 1246.

In [6]:
def find_best_c(response, predictor):
    # grid for predictor and response (beta0 and beta1)
    beta0_grid = np.arange(-25, -15, 0.1)
    beta1_grid = np.arange(-5, 5, 0.01)

    # loop over grid of c values finding rmse for each variable
    rmse_grid = [(b0, b1, calc_rmse(response, b0 + b1 * predictor))
           for b0 in beta0_grid
           for b1 in beta1_grid]

    # determine which vector c gives the smallest rmse
    best_b0, best_b1, min_rmse = min(rmse_grid, key=lambda x: x[2])

    # report best vector c as a prediction
    print(f"The best values for beta0 and beta1 are ({best_b0}, {best_b1} with an RMSE of {min_rmse}")

    # return best values of beta0 and beta1
    return best_b0, best_b1

In [7]:
# Find best coefficients using your data
beta0, beta1 = find_best_c(air_quality["C6H6(GT)"], air_quality["PT08.S1(CO)"])

# Predict C6H6(GT) for new PT08.S1(CO) values
new_values = np.array([946, 1075, 1246])
predictions = beta0 + beta1 * new_values

print(predictions)

The best values for beta0 and beta1 are (-22.99999999999997, 0.02999999999989278 with an RMSE of 3.5429053908306787
[ 5.38  9.25 14.38]


Gradient Descent Pseudo-code:

1. Create a squared error loss function that takes in y and c and outputs $(y - c)^2$.
2. Create a root mean squared error (objective) function that takes in y and c and outputs $RMSE(c) = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y-c)^2}$
3. Create a difference quotient function to approximate the slope of the tangent line that takes in the guess c, a small change to c - I'll call it $\Delta$, and y that outputs

$$diff\_quotient = \frac{RMSE(c + \Delta) - RMSE(c)}{\Delta}$$

4. Pick a starting value for cur_c.
5. Evaluate the difference quotient at cur_c.
6. Update the value by moving (a small step) in the negative direction of the difference quotient new_c = cur_c - diff_quotient * step_size
7. Check if abs(new_c - cur_c) < num_tol where num_tol is a small value.
   - If so, update the cur_c to be the new_c value and stop.
   - If not, update the value of cur_c to new_c and repeat steps 5-7.
     - Put in a safety that stops the loop after a maximum number of iterations is reached (even if abs(new_c - cur_c) < num_tol` is not met)
8. Use the last value as the prediction!
9. Wrap the above into a function!

- Run your algorithm on the C6H6(GT) variable and determine the optimal prediction. Use 0 as your starting point.

  - Note: My $\Delta$ was 0.001, my step size was 0.01, and my tolerance was 0.0001. It took around 4000 iterations for my algorithm to converge.

- Just to make sure your algorithm generalizes, run it using the PT08.S1(CO) variable as the *response* (this shouldn't involve any new functions, just new function calls!). Use 1100 as your starting point.

  - Note: My $\Delta$ was 0.001, my step size was 0.1, and my tolerance was 0.0001. It took around 8500 iterations for my algorithm to converge.

In [29]:
def calc_rmse(response, c):
    return np.sqrt(1/len(response) * sum((response-c)**2))

def diff_quotient(response, cur_c, delta):
    return (calc_rmse(response, cur_c + delta) - calc_rmse(response, cur_c)) / delta

def gradient_descent(response, cur_c=0, delta=0.001, step_size=0.01, tol=0.0001, max_iter=10000, debug=False):
    for i in range(max_iter):
        dq = diff_quotient(response, cur_c, delta)
        new_c = cur_c - dq * step_size
        if debug:
            print(f"Iter {i}: cur_c={cur_c}, dq={dq}, new_c={new_c}")
        if abs(new_c - cur_c) < tol:
            return new_c
        else:
          cur_c = new_c

    print(f"Warning: max iterations ({max_iter}) reached without convergence.")
    return cur_c

In [ ]:
print(gradient_descent(air_quality["C6H6(GT)"], cur_c=0, delta=0.001, step_size=0.01, tol=0.0001))
print(gradient_descent(air_quality["PT08.S1(CO)"], cur_c=1100, delta=0.001, step_size=0.1, tol=0.0001))

# Using $y$ and another numeric variable $x$

Next, you'll implement the gradient descent idea to find the optimal value of $b_0$ and $b_1$ using PT08.S1(CO) as your $x$ variable and C6H6(GT) as your $y$ variable. The pseudo code is very similar to that above, but you have two variables to optimize over ($b_0$ and $b_1$) instead of one ($c$).

## Gradient Descent Pseudo-code

1. Create a squared error loss function that takes in $y$, $x$, $b_0$, and $b_1$ that outputs $(y - b_0 - b_1 x)^2$

2. Create a mean squared error (objective) function that takes in the same values that outputs $RMSE(b_0, b_1) = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y - b_0 - b_1 x)^2}$

3. Create a difference quotient function for $b_0$ to approximate the slope of the tangent line in the direction of $b_0$. It should take in $y$, $x$, $b_0$, a small change to $b_0$ — I'll call it $\Delta_0$, and $b_1$ that outputs

$$diff\_quotient\_b0 = \frac{RMSE(b_0 + \Delta_0, b_1) - RMSE(b_0, b_1)}{\Delta_0}$$

4. Create a difference quotient function for $b_1$ to approximate the slope of the tangent line in the direction of $b_1$. It should take in $y$, $x$, $b_0$, $b_1$ and a small change to $b_1$ — I'll call it $\Delta_1$ that outputs

$$diff\_quotient\_b1 = \frac{RMSE(b_0, b_1 + \Delta_1) - RMSE(b_0, b_1)}{\Delta_1}$$

5. Pick starting values for b_0 and b_1 (say cur_b0 and cur_b1).

6. Evaluate the difference quotient for b_0 at the cur_b0 and cur_b1 values.

7. Update the value of b_0 by moving (a small step) in the negative direction of the difference quotient: new_b0 = cur_b0 - diff_quotient_b0 * step_size_b0

8. Evaluate the difference quotient for b_1 at the new_b0 and cur_b1 values.

9. Update the value of b_1 by moving (a small step) in the negative direction of the difference quotient: new_b1 = cur_b1 - diff_quotient_b1 * step_size_b1

10. Check if the distance between the (cur_b0, cur_b1) vector to the (new_b0, new_b1) vector is less than some small tolerance. (Hint: Look up the euclidean distance between two vectors, there is a function in numpy to calculate it — or just code it up yourself).
    - If so, update the cur_b0 and cur_b1 to be the new_b0 and new_b1 values and stop.
    - If not, update the cur_b0 and cur_b1 to be the new_b0 and new_b1 values and repeat steps 6–10.
    - Put in a safety that stops the loop after a maximum number of iterations is reached (even if the tolerance is not met).

11. Use the last values as the estimates for $b_0$ and $b_1$!

In [37]:
def calc_rmse(response, predictor, b0, b1):
    return np.sqrt(np.mean((response - b0 - b1 * predictor) ** 2))

def diff_quotient_b0(response, predictor, b0, b1, delta0,):
    return (calc_rmse(response, predictor, b0 + delta0, b1) - calc_rmse(response, predictor, b0, b1)) / delta0

def diff_quotient_b1(response, predictor, b0, b1, delta1):
    return (calc_rmse(response, predictor, b0, b1 + delta1) - calc_rmse(response, predictor, b0, b1)) / delta1

def gradient_descent2(response, predictor, cur_b0=-20, cur_b1=0,
                     delta0=0.005, delta1=0.005,
                     step_size_b0=0.5, step_size_b1=0.00005,
                     tol=0.0001, max_iter=100000, debug=False):
    for i in range(max_iter):
        # update b0
        dq0 = diff_quotient_b0(response, predictor, cur_b0, cur_b1, delta0)
        new_b0 = cur_b0 - dq0 * step_size_b0

        # update b1 using new_b0
        dq1 = diff_quotient_b1(response, predictor, new_b0, cur_b1, delta1)
        new_b1 = cur_b1 - dq1 * step_size_b1

        if debug:
            print(f"Iter {i}: b0={new_b0}, b1={new_b1}, dq={dq0}, dq1={dq1}")

        # check Euclidean distance
        dist = np.linalg.norm(np.array([new_b0, new_b1]) - np.array([cur_b0, cur_b1]))

        if dist < tol:
            return new_b0, new_b1
        else:
          cur_b0, cur_b1 = new_b0, new_b1

    print(f"Warning: max iterations ({max_iter}) reached without convergence.")
    return cur_b0, cur_b1

Use initial starting values of -20 for the intercept and 0 for the slope. Report your optimal $b_0$ and $b_1$ combination. Then use these values to predict a new C6H6(GT) for a PT08.S1(CO) of 946, 1075, and 1246.

- Note: I used a step size of 0.5 for the intercept, 0.00005 for the slope, a delta of 0.005 for both the intercept and slope, and a tolerance of 0.0001.

- I used 100,000 iterations and my algorithm took 5–10 minutes to run. I hit my limit and got reasonably close to the truth but not super close (−23.18 and 0.0017, respectively).
- That's close enough here! We'd need to do some fancier adaptive step size to make this work better.

In [40]:
beta0, beta1 = gradient_descent2(air_quality["C6H6(GT)"], air_quality["PT08.S1(CO)"],
                  cur_b0=-20, cur_b1=0,
                  delta0=0.005, delta1=0.005,
                  step_size_b0=0.5, step_size_b1=0.00005,
                  tol=0.0001, max_iter=100000, debug=False)

In [39]:
print((beta0, beta1))

(np.float64(-22.30482975822852), np.float64(-0.0014903666811793767))


In [41]:
# Predict C6H6(GT) for new PT08.S1(CO) values
new_values = np.array([946, 1075, 1246])
predictions = beta0 + beta1 * new_values

print(predictions)

[-23.71471664 -23.90697394 -24.16182664]
